In [1]:
import sys
from pathlib import Path

import yaml

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from training.model_factory import create_model

CONFIG_DIR = REPO_ROOT / "configs" / "heterogeneous_dataset"

In [2]:
CONFIG_STEMS = [
    # FNO
    "fno_m8x32x16_h64_hetero",
    "fno_m4x16x8_h128_hetero",
    # U-FNO
    "ufno_hetero",
    # UNet
    "unet_hetero",
    # UNO
    "uno_hetero",
    # LOGLO
    "modulated_loglo_hetero",
    "vanilla_loglo_hetero",
]


def load_config(stem: str) -> dict:
    with open(CONFIG_DIR / f"{stem}.yml") as f:
        return yaml.safe_load(f)


def count_params(model) -> int:
    return sum(p.numel() for p in model.parameters())

In [3]:
rows = []
for stem in CONFIG_STEMS:
    cfg = load_config(stem)
    model_cfg = cfg["model"]
    model_type = model_cfg["type"]
    model = create_model(model_cfg, model_type)
    n_params = count_params(model)
    variant = "hetero" if cfg["data"].get("heterogeneous") else "homo"
    rows.append((stem, model_type, variant, n_params))
    print(f"{stem:40s}  type={model_type:16s}  params={n_params:,}")

print()
print(f"{'Architecture':40s}  {'Params':>15s}")
print("-" * 58)
for stem, model_type, variant, n_params in rows:
    print(f"{stem:40s}  {n_params:>15,}")

fno_m8x32x16_h64_hetero                   type=fno               params=37,801,284
fno_m4x16x8_h128_hetero                   type=fno               params=26,451,012
ufno_hetero                               type=ufno              params=41,956,036
unet_hetero                               type=unet3d            params=423,537,604
fno_skip='linear'
channel_mlp_skip='linear'
fno_skip='linear'
channel_mlp_skip='linear'
fno_skip='linear'
channel_mlp_skip='linear'
fno_skip='linear'
channel_mlp_skip='linear'
fno_skip='linear'
channel_mlp_skip='linear'
uno_hetero                                type=uno               params=92,691,780
modulated_loglo_hetero                    type=modulated_loglo   params=14,102,148
vanilla_loglo_hetero                      type=vanilla_loglo     params=13,753,092

Architecture                                       Params
----------------------------------------------------------
fno_m8x32x16_h64_hetero                        37,801,284
fno_m4x16x8_h128_heter